# 📊 Executive Dashboard — Ecommerce Retail Lakehouse
**Unified KPIs across all 6 Gold tables (Sales · Customers · Products · Inventory · Payments · Reviews)**

> **Before running:** ensure `config.py` has valid `ACCESS_KEY`, `SECRET_KEY`, `SESSION_TOKEN`, `S3_GOLD_BASE`.

In [ ]:
exec(open('/Workspace/ecommerce_retail/config.py').read())

import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings; warnings.filterwarnings('ignore')

def read_gold(t):
    return (spark.read
        .option('fs.s3a.access.key', ACCESS_KEY)
        .option('fs.s3a.secret.key', SECRET_KEY)
        .option('fs.s3a.session.token', SESSION_TOKEN)
        .option('fs.s3a.aws.credentials.provider',
                'org.apache.hadoop.fs.s3a.TemporaryAWSCredentialsProvider')
        .parquet(f'{S3_GOLD_BASE}/{t}/')
        .toPandas())

print('✅ Setup complete')

In [ ]:
print('⏳ Loading Gold tables...')
df_sales     = read_gold('sales_summary')
df_customers = read_gold('customer_analytics')
df_products  = read_gold('product_performance')
df_inventory = read_gold('inventory_health')
df_payments  = read_gold('payment_summary')
df_reviews   = read_gold('review_analytics')
print(f'✅ sales_summary       : {len(df_sales):,} rows')
print(f'✅ customer_analytics  : {len(df_customers):,} rows')
print(f'✅ product_performance : {len(df_products):,} rows')
print(f'✅ inventory_health    : {len(df_inventory):,} rows')
print(f'✅ payment_summary     : {len(df_payments):,} rows')
print(f'✅ review_analytics    : {len(df_reviews):,} rows')

In [ ]:
# ── Executive KPI Summary ─────────────────────────────────────────────────
total_net_rev    = df_sales['net_revenue'].sum()
total_orders     = df_sales['total_orders'].sum()
avg_aov          = df_sales['avg_order_value'].mean()
avg_fulfill      = df_sales['avg_fulfillment_days'].mean()
total_discounts  = df_sales['total_discounts'].sum()
total_customers  = len(df_customers)
avg_ltv          = df_customers['lifetime_value'].mean()
avg_loyalty      = df_customers['loyalty_points'].mean()
active_cust      = (df_customers['customer_status'] == 'Active').sum()
total_products   = len(df_products)
total_profit     = df_products['profit_estimate'].sum()
avg_prod_rating  = df_products['avg_review_rating'].mean()
total_units_sold = df_products['units_sold'].sum()
total_sku        = len(df_inventory)
low_stock_count  = df_inventory['low_stock_flag'].sum()
low_stock_pct    = low_stock_count / total_sku * 100
total_inv_value  = df_inventory['inventory_value'].sum()
success_txns     = df_payments[df_payments['payment_status']=='Success']['transaction_count'].sum()
total_txns       = df_payments['transaction_count'].sum()
pay_success_rate = success_txns / total_txns * 100
avg_fraud        = df_payments['avg_fraud_score'].mean()
avg_rating_rev   = df_reviews['avg_rating'].mean()
promoter_pct     = (df_reviews[df_reviews['nps_category']=='Promoter']['total_reviews'].sum()
                    / df_reviews['total_reviews'].sum() * 100)
positive_pct     = df_reviews['positive_rate_pct'].mean()

metrics = [
    '💰  Total Net Revenue','🛒  Total Orders','💳  Avg Order Value','🚚  Avg Fulfillment (days)',
    '🏷️  Total Discounts','👥  Total Customers','✅  Active Customers',
    '⭐  Avg Lifetime Value','🎯  Avg Loyalty Points','📦  Total Products',
    '📈  Total Profit','⭐  Avg Product Rating','🛍️  Total Units Sold',
    '🏭  Total SKUs','⚠️  Low Stock SKUs','🏦  Total Inventory Value',
    '✅  Payment Success Rate','🔐  Avg Fraud Score',
    '⭐  Avg Review Rating','😊  Promoter %','👍  Positive Sentiment %'
]
values = [
    f'${total_net_rev:,.0f}', f'{total_orders:,}', f'${avg_aov:,.0f}', f'{avg_fulfill:.1f}',
    f'${total_discounts:,.0f}', f'{total_customers:,}', f'{active_cust:,}',
    f'${avg_ltv:,.0f}', f'{avg_loyalty:,.0f}', f'{total_products:,}',
    f'${total_profit:,.0f}', f'{avg_prod_rating:.2f}', f'{total_units_sold:,}',
    f'{total_sku:,}', f'{low_stock_count:,} ({low_stock_pct:.1f}%)', f'${total_inv_value:,.0f}',
    f'{pay_success_rate:.1f}%', f'{avg_fraud:.2f}',
    f'{avg_rating_rev:.2f}/5.0', f'{promoter_pct:.1f}%', f'{positive_pct:.1f}%'
]
sources = [
    'Sales Summary','Sales Summary','Sales Summary','Sales Summary','Sales Summary',
    'Customer Analytics','Customer Analytics','Customer Analytics','Customer Analytics',
    'Product Performance','Product Performance','Product Performance','Product Performance',
    'Inventory Health','Inventory Health','Inventory Health',
    'Payment Summary','Payment Summary',
    'Review Analytics','Review Analytics','Review Analytics'
]
row_colors = [['#f0f4ff' if i%2==0 else '#fff' for i in range(len(metrics))]]*3
fig = go.Figure(data=[go.Table(
    columnwidth=[320,200,200],
    header=dict(values=['<b>KPI</b>','<b>Value</b>','<b>Source</b>'],
                fill_color='#1a1a2e', font=dict(color='white',size=13), align='left', height=36),
    cells=dict(values=[metrics,values,sources], fill_color=row_colors,
               font=dict(color='#1a1a2e',size=12), align='left', height=30)
)])
fig.update_layout(title='📊 Executive KPI Summary — All 6 Gold Tables', height=680,
                  margin=dict(l=10,r=10,t=50,b=10))
fig.show()

In [ ]:
# ── Monthly Revenue & Orders Trend ────────────────────────────────────────
df_m = (df_sales.groupby(['order_year','order_month'])
        .agg(net_revenue=('net_revenue','sum'),
             total_orders=('total_orders','sum'),
             total_discounts=('total_discounts','sum'),
             avg_aov=('avg_order_value','mean'))
        .reset_index().sort_values(['order_year','order_month']))
df_m['period'] = df_m['order_year'].astype(str)+'-'+df_m['order_month'].astype(str).str.zfill(2)

fig = make_subplots(rows=2,cols=2,
    subplot_titles=['Monthly Net Revenue ($)','Monthly Orders',
                    'Monthly Discounts ($)','Avg Order Value ($)'],
    vertical_spacing=0.18, horizontal_spacing=0.1)
fig.add_trace(go.Scatter(x=df_m['period'],y=df_m['net_revenue'],mode='lines+markers',
    name='Net Revenue',line=dict(color='#3498db',width=2),
    fill='tozeroy',fillcolor='rgba(52,152,219,0.1)'),row=1,col=1)
fig.add_trace(go.Bar(x=df_m['period'],y=df_m['total_orders'],
    name='Orders',marker_color='#2ecc71'),row=1,col=2)
fig.add_trace(go.Scatter(x=df_m['period'],y=df_m['total_discounts'],mode='lines+markers',
    name='Discounts',line=dict(color='#e74c3c',width=2)),row=2,col=1)
fig.add_trace(go.Scatter(x=df_m['period'],y=df_m['avg_aov'],mode='lines+markers',
    name='AOV',line=dict(color='#9b59b6',width=2)),row=2,col=2)
fig.update_layout(title='📈 Sales Performance Trends',height=550,showlegend=False)
fig.show()

In [ ]:
# ── Customer Analytics ────────────────────────────────────────────────────
SEG_COLORS = {'Bronze':'#cd7f32','Silver':'#a8a9ad','Gold':'#ffd700','Platinum':'#e5e4e2'}
seg_counts = df_customers['customer_segment'].value_counts().reset_index()
seg_counts.columns = ['segment','count']
seg_ltv = df_customers.groupby('customer_segment')['lifetime_value'].mean().sort_values().reset_index()
country_ltv = df_customers.groupby('country')['lifetime_value'].mean().sort_values(ascending=False).reset_index()
seg_country = df_customers.groupby(['country','customer_segment']).size().reset_index(name='count')

fig = make_subplots(rows=2,cols=2,
    subplot_titles=['Segment Mix','Avg LTV by Segment ($)','Avg LTV by Country ($)','Segment by Country'],
    specs=[[{'type':'pie'},{'type':'bar'}],[{'type':'bar'},{'type':'bar'}]],
    vertical_spacing=0.18,horizontal_spacing=0.1)
fig.add_trace(go.Pie(labels=seg_counts['segment'],values=seg_counts['count'],hole=0.45,
    marker=dict(colors=[SEG_COLORS.get(s,'#888') for s in seg_counts['segment']])),row=1,col=1)
fig.add_trace(go.Bar(x=seg_ltv['lifetime_value'],y=seg_ltv['customer_segment'],orientation='h',
    text=seg_ltv['lifetime_value'].apply(lambda x: f'${x:,.0f}'),textposition='outside',
    marker_color=[SEG_COLORS.get(s,'#888') for s in seg_ltv['customer_segment']]),row=1,col=2)
fig.add_trace(go.Bar(x=country_ltv['country'],y=country_ltv['lifetime_value'],
    text=country_ltv['lifetime_value'].apply(lambda x: f'${x:,.0f}'),textposition='outside',
    marker_color='#3498db'),row=2,col=1)
for seg in ['Bronze','Silver','Gold','Platinum']:
    d = seg_country[seg_country['customer_segment']==seg]
    if not d.empty:
        fig.add_trace(go.Bar(name=seg,x=d['country'],y=d['count'],
            marker_color=SEG_COLORS.get(seg,'#888'),showlegend=True),row=2,col=2)
fig.update_layout(title='👥 Customer Analytics Overview',height=620,barmode='stack')
fig.show()

In [ ]:
# ── Product Performance ───────────────────────────────────────────────────
cat_m = (df_products.groupby('category')
         .agg(revenue=('revenue','sum'),profit=('profit_estimate','sum'),
              units_sold=('units_sold','sum'),avg_rating=('avg_review_rating','mean'))
         .reset_index().sort_values('revenue',ascending=False))
cat_m['margin_pct'] = cat_m['profit']/cat_m['revenue']*100
top10 = df_products.nlargest(10,'revenue').sort_values('revenue',ascending=True)

fig = make_subplots(rows=2,cols=2,
    subplot_titles=['Revenue by Category','Profit Margin % by Category',
                    'Units Sold by Category','Top 10 Products by Revenue'],
    specs=[[{'type':'bar'},{'type':'bar'}],[{'type':'bar'},{'type':'bar'}]],
    vertical_spacing=0.18,horizontal_spacing=0.1)
fig.add_trace(go.Bar(y=cat_m['category'],x=cat_m['revenue'],orientation='h',
    marker_color='#3498db',text=cat_m['revenue'].apply(lambda x:f'${x/1e6:.1f}M'),
    textposition='outside'),row=1,col=1)
fig.add_trace(go.Bar(y=cat_m['category'],x=cat_m['margin_pct'],orientation='h',
    marker_color='#2ecc71',text=cat_m['margin_pct'].apply(lambda x:f'{x:.1f}%'),
    textposition='outside'),row=1,col=2)
fig.add_trace(go.Bar(y=cat_m['category'],x=cat_m['units_sold'],orientation='h',
    marker_color='#9b59b6',text=cat_m['units_sold'].apply(lambda x:f'{x:,}'),
    textposition='outside'),row=2,col=1)
fig.add_trace(go.Bar(y=top10['product_name'],x=top10['revenue'],orientation='h',
    marker_color='#f39c12',text=top10['revenue'].apply(lambda x:f'${x/1e3:.0f}K'),
    textposition='outside'),row=2,col=2)
fig.update_layout(title='📦 Product Performance Overview',height=620,showlegend=False)
fig.show()

In [ ]:
# ── Inventory Health ──────────────────────────────────────────────────────
inv_st = df_inventory['inventory_status'].value_counts().reset_index()
inv_st.columns = ['status','count']
low_ctry = (df_inventory[df_inventory['low_stock_flag']==True]
            .groupby('warehouse_country').size().reset_index(name='n')
            .sort_values('n',ascending=False))
sor = (df_inventory[df_inventory['available_stock']<df_inventory['reorder_point']]
       .groupby('warehouse_country').size().reset_index(name='n')
       .sort_values('n',ascending=False))
inv_val = (df_inventory.groupby('warehouse_country')['inventory_value'].sum()
           .reset_index().sort_values('inventory_value',ascending=False))
SC = {'in_stock':'#2ecc71','low_stock':'#f39c12','out_of_stock':'#e74c3c'}

fig = make_subplots(rows=2,cols=2,
    subplot_titles=['Inventory Status','Low Stock by Country',
                    'Below Reorder Point','Inventory Value by Country ($)'],
    specs=[[{'type':'pie'},{'type':'bar'}],[{'type':'bar'},{'type':'bar'}]],
    vertical_spacing=0.18,horizontal_spacing=0.1)
fig.add_trace(go.Pie(labels=inv_st['status'],values=inv_st['count'],hole=0.45,
    marker=dict(colors=[SC.get(s,'#888') for s in inv_st['status']])),row=1,col=1)
fig.add_trace(go.Bar(x=low_ctry['warehouse_country'],y=low_ctry['n'],
    marker_color='#f39c12',text=low_ctry['n'],textposition='outside'),row=1,col=2)
fig.add_trace(go.Bar(x=sor['warehouse_country'],y=sor['n'],
    marker_color='#e74c3c',text=sor['n'],textposition='outside'),row=2,col=1)
fig.add_trace(go.Bar(x=inv_val['warehouse_country'],y=inv_val['inventory_value'],
    marker_color='#3498db',
    text=inv_val['inventory_value'].apply(lambda x:f'${x/1e6:.1f}M'),
    textposition='outside'),row=2,col=2)
fig.update_layout(title='🏭 Inventory Health Overview',height=620,showlegend=False)
fig.show()

In [ ]:
# ── Payment Summary ───────────────────────────────────────────────────────
ppiv = (df_payments.groupby(['payment_method','payment_status'])['transaction_count'].sum()
        .reset_index().pivot(index='payment_method',columns='payment_status',values='transaction_count')
        .fillna(0).reset_index())
fraud_m = (df_payments.groupby('payment_method')['avg_fraud_score'].mean()
           .reset_index().sort_values('avg_fraud_score',ascending=False))
vol_m   = (df_payments.groupby('payment_method')['total_amount'].sum()
           .reset_index().sort_values('total_amount',ascending=False))
ov_st   = df_payments.groupby('payment_status')['transaction_count'].sum().reset_index()
SPC = {'Success':'#2ecc71','Failed':'#e74c3c','Refunded':'#f39c12','Pending':'#3498db'}

fig = make_subplots(rows=2,cols=2,
    subplot_titles=['Status by Payment Method','Avg Fraud Score by Method',
                    'Total Volume ($) by Method','Overall Status Mix'],
    specs=[[{'type':'bar'},{'type':'bar'}],[{'type':'bar'},{'type':'pie'}]],
    vertical_spacing=0.18,horizontal_spacing=0.1)
for st in ['Success','Failed','Refunded','Pending']:
    if st in ppiv.columns:
        fig.add_trace(go.Bar(name=st,x=ppiv['payment_method'],y=ppiv[st],
            marker_color=SPC.get(st,'#888')),row=1,col=1)
fig.add_trace(go.Bar(x=fraud_m['payment_method'],y=fraud_m['avg_fraud_score'],
    marker=dict(color=fraud_m['avg_fraud_score'],colorscale='Reds',showscale=False),
    text=fraud_m['avg_fraud_score'].round(2),textposition='outside'),row=1,col=2)
fig.add_trace(go.Bar(x=vol_m['payment_method'],y=vol_m['total_amount'],
    marker_color='#3498db',
    text=vol_m['total_amount'].apply(lambda x:f'${x/1e6:.1f}M'),textposition='outside'),row=2,col=1)
fig.add_trace(go.Pie(labels=ov_st['payment_status'],values=ov_st['transaction_count'],hole=0.45,
    marker=dict(colors=[SPC.get(s,'#888') for s in ov_st['payment_status']])),row=2,col=2)
fig.update_layout(title='💳 Payment Summary Overview',height=620,barmode='stack')
fig.show()

In [ ]:
# ── Review Analytics ──────────────────────────────────────────────────────
df_rp = df_reviews.drop_duplicates(subset=['product_id']).copy()
df_ru = df_reviews.drop_duplicates(subset=['review_id']).copy() if 'review_id' in df_reviews.columns else df_reviews.copy()
cat_r  = df_rp.groupby('category')['avg_rating'].mean().reset_index().sort_values('avg_rating',ascending=True)
sent_c = df_ru['sentiment'].value_counts().reset_index(); sent_c.columns=['s','c']
nps_c  = df_ru['nps_category'].value_counts().reset_index(); nps_c.columns=['n','c']
trd_c  = df_ru['rating_trend'].value_counts().reset_index(); trd_c.columns=['t','c']
SC2 = {'Positive':'#2ecc71','Neutral':'#3498db','Negative':'#e74c3c'}
NC  = {'Promoter':'#2ecc71','Passive':'#f39c12','Detractor':'#e74c3c'}
TC  = {'Improving':'#2ecc71','Stable':'#3498db','Declining':'#e74c3c'}

fig = make_subplots(rows=2,cols=2,
    subplot_titles=['Avg Rating by Category','Sentiment Distribution',
                    'NPS Categories','Rating Trend'],
    specs=[[{'type':'bar'},{'type':'pie'}],[{'type':'pie'},{'type':'pie'}]],
    vertical_spacing=0.18,horizontal_spacing=0.1)
fig.add_trace(go.Bar(y=cat_r['category'],x=cat_r['avg_rating'],orientation='h',
    marker=dict(color=cat_r['avg_rating'],colorscale='RdYlGn',cmin=1,cmax=5,
                colorbar=dict(title='Rating',x=0.46)),
    text=cat_r['avg_rating'].round(2),textposition='outside'),row=1,col=1)
fig.add_trace(go.Pie(labels=sent_c['s'],values=sent_c['c'],hole=0.45,
    marker=dict(colors=[SC2.get(s,'#888') for s in sent_c['s']])),row=1,col=2)
fig.add_trace(go.Pie(labels=nps_c['n'],values=nps_c['c'],hole=0.45,
    marker=dict(colors=[NC.get(s,'#888') for s in nps_c['n']])),row=2,col=1)
fig.add_trace(go.Pie(labels=trd_c['t'],values=trd_c['c'],hole=0.45,
    marker=dict(colors=[TC.get(s,'#888') for s in trd_c['t']])),row=2,col=2)
fig.update_layout(title='⭐ Review Analytics Overview',height=620)
fig.show()